<a href="https://colab.research.google.com/github/rutujahingmire8497-code/Keyword-and-Topic-Extraction-from-Research-Ppaers/blob/main/RUTUJA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===================== SETTINGS =====================
N_CNN = 100
N_ARXIV = 50
LAUNCH_DEMO = True
# ====================================================

!pip install -q "datasets<4" transformers sentencepiece rouge-score networkx scikit-learn gradio

import re, json, time, zipfile
from statistics import mean
import numpy as np
import networkx as nx
import torch
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ---------------- Preprocessing (removes personal info) ----------------
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
PHONE_RE = re.compile(r"(\+?\d[\d\s().-]{8,}\d)")
HANDLE_RE = re.compile(r"(?<!\w)@\w{2,}")
SPACE_RE = re.compile(r"\s+")
ABBREV = ["Mr", "Mrs", "Ms", "Dr", "Prof", "Sr", "Jr", "St", "vs", "etc", "e.g", "i.e", "Fig", "al", "No"]
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[\"'A-Z0-9])")

def clean_text(text):
    for rx in (EMAIL_RE, URL_RE, PHONE_RE, HANDLE_RE):
        text = rx.sub(" ", text)
    return SPACE_RE.sub(" ", text.replace("\n", " ")).strip()

def split_sentences(text):
    for a in ABBREV:
        text = text.replace(f"{a}. ", f"{a}<DOT> ")
    parts = [p.replace("<DOT>", ".").strip() for p in SENT_SPLIT_RE.split(text)]
    return [p for p in parts if len(p.split()) >= 3]

# ---------------- Baseline: TextRank (extractive) ----------------
def textrank_summarize(text, num_sentences=3):
    sentences = split_sentences(clean_text(text))
    if len(sentences) <= num_sentences:
        return " ".join(sentences)
    try:
        tfidf = TfidfVectorizer(stop_words="english").fit_transform(sentences)
    except ValueError:
        return " ".join(sentences[:num_sentences])
    sim = cosine_similarity(tfidf)
    np.fill_diagonal(sim, 0.0)
    graph = nx.from_numpy_array(sim)
    try:
        scores = nx.pagerank(graph, max_iter=200)
    except nx.PowerIterationFailedConvergence:
        scores = {i: float(sim[i].sum()) for i in range(len(sentences))}
    top = sorted(scores, key=scores.get, reverse=True)[:num_sentences]
    return " ".join(sentences[i] for i in sorted(top))

# ---------------- Advanced: BART / T5 (abstractive) ----------------
MODEL_NAMES = {"bart": "facebook/bart-large-cnn", "t5": "t5-small"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: GPU is OFF. Runtime > Change runtime type > T4 GPU, then run again.")

class AbstractiveSummarizer:
    def __init__(self, key="bart"):
        self.name = MODEL_NAMES[key]
        self.tok = AutoTokenizer.from_pretrained(self.name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(self.name).to(DEVICE).eval()
        self.prefix = "summarize: " if "t5" in self.name else ""

    @torch.no_grad()
    def summarize(self, text, max_input_tokens=1024, max_length=128, min_length=30, num_beams=4):
        text = self.prefix + clean_text(text)
        inputs = self.tok(text, return_tensors="pt", truncation=True, max_length=max_input_tokens).to(DEVICE)
        ids = self.model.generate(**inputs, max_length=max_length, min_length=min_length,
                                  num_beams=num_beams, length_penalty=2.0,
                                  no_repeat_ngram_size=3, early_stopping=True)
        return self.tok.decode(ids[0], skip_special_tokens=True).strip()

_models = {}
def get_model(key):
    if key not in _models:
        _models[key] = AbstractiveSummarizer(key)
    return _models[key]

# ---------------- ROUGE ----------------
_scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def rouge_single(ref, pred):
    return {k: round(v.fmeasure * 100, 2) for k, v in _scorer.score(ref, pred).items()}

def rouge_corpus(refs, preds):
    rows = [rouge_single(r, p) for r, p in zip(refs, preds)]
    return {k: round(mean(r[k] for r in rows), 2) for k in ("rouge1", "rouge2", "rougeL")}

# ---------------- Data ----------------
DATASETS = {
    "cnn_dailymail": dict(path="abisee/cnn_dailymail", name="3.0.0", text="article", summary="highlights"),
    "arxiv": dict(path="ccdv/arxiv-summarization", name=None, text="article", summary="abstract"),
}

def load_samples(dataset="cnn_dailymail", split="test", n=100, seed=42):
    cfg = DATASETS[dataset]
    kw = {"name": cfg["name"]} if cfg["name"] else {}
    ds = load_dataset(cfg["path"], split=split, trust_remote_code=True, **kw)
    ds = ds.shuffle(seed=seed).select(range(min(n, len(ds))))
    return [clean_text(x[cfg["text"]]) for x in ds], [clean_text(x[cfg["summary"]]) for x in ds]

# ---------------- Experiment runner ----------------
def avg_words(texts):
    return round(mean(len(t.split()) for t in texts), 1)

def run_experiment(dataset="cnn_dailymail", n=100, models=("bart", "t5"), num_sentences=3):
    docs, refs = load_samples(dataset, n=n)
    results, outputs = {}, {"reference": refs}

    t0 = time.time()
    preds = [textrank_summarize(d, num_sentences) for d in docs]
    results["TextRank (baseline)"] = {**rouge_corpus(refs, preds),
                                      "seconds": round(time.time() - t0, 1), "words": avg_words(preds)}
    outputs["textrank"] = preds

    for key in models:
        m = get_model(key)
        t0 = time.time()
        preds = [m.summarize(d) for d in docs]
        results[f"{key.upper()} ({m.name})"] = {**rouge_corpus(refs, preds),
                                                "seconds": round(time.time() - t0, 1), "words": avg_words(preds)}
        outputs[key] = preds

    print(f"\nDataset: {dataset} | documents: {len(docs)} | avg reference words: {avg_words(refs)}")
    print(f"{'Model':42s} {'R-1':>6s} {'R-2':>6s} {'R-L':>6s} {'sec':>7s} {'words':>7s}")
    for name, r in results.items():
        print(f"{name:42s} {r['rouge1']:6.2f} {r['rouge2']:6.2f} {r['rougeL']:6.2f} {r['seconds']:7.1f} {r['words']:7.1f}")

    json.dump(results, open(f"{dataset}_scores.json", "w"), indent=2)
    json.dump(outputs, open(f"{dataset}_outputs.json", "w"), indent=2)
    return results, outputs

# ---------------- RUN EXPERIMENTS ----------------
saved_files, all_results = [], {}
if N_CNN > 0:
    res_cnn, out_cnn = run_experiment("cnn_dailymail", n=N_CNN)
    all_results["CNN/DailyMail"] = res_cnn
    saved_files += ["cnn_dailymail_scores.json", "cnn_dailymail_outputs.json"]

if N_ARXIV > 0:
    try:
        res_arxiv, out_arxiv = run_experiment("arxiv", n=N_ARXIV)
        all_results["arXiv"] = res_arxiv
        saved_files += ["arxiv_scores.json", "arxiv_outputs.json"]
    except Exception as e:
        print("\n!!! arXiv run failed (CNN results are safe). Error:", repr(e)[:500])

# ---------------- README TABLE (copy-paste into GitHub README) ----------------
md = ["| Dataset | Model | ROUGE-1 | ROUGE-2 | ROUGE-L | Time (s) | Avg words |", "|---|---|---|---|---|---|---|"]
for ds_name, res in all_results.items():
    for model_name, r in res.items():
        md.append(f"| {ds_name} | {model_name} | {r['rouge1']} | {r['rouge2']} | {r['rougeL']} | {r['seconds']} | {r['words']} |")
md_text = "\n".join(md)
print("\n===== COPY THIS TABLE INTO YOUR README / REPORT =====\n")
print(md_text)
open("results_table.md", "w").write(md_text)
saved_files.append("results_table.md")

# ---------------- EXAMPLE SUMMARIES (for error analysis) ----------------
if N_CNN > 0:
    print("\n\n===== EXAMPLE SUMMARIES (CNN/DailyMail) =====")
    for i in range(3):
        print(f"\n--- Example {i+1} ---")
        print("REFERENCE:", out_cnn["reference"][i])
        print("TEXTRANK :", out_cnn["textrank"][i])
        print("BART     :", out_cnn["bart"][i])
        print("T5       :", out_cnn["t5"][i])

# ---------------- DOWNLOAD ONE ZIP ----------------
with zipfile.ZipFile("results.zip", "w") as z:
    for f in saved_files:
        z.write(f)
from google.colab import files
files.download("results.zip")

# ---------------- DEMO (summarizer tool) ----------------
if LAUNCH_DEMO:
    import gradio as gr

    def summarize_ui(text, method, n_sent, max_len, reference):
        if not text.strip():
            return "Please paste some text."
        outs = {}
        if method in ("TextRank", "Compare all"):
            outs["TextRank"] = textrank_summarize(text, int(n_sent))
        if method in ("BART", "Compare all"):
            outs["BART"] = get_model("bart").summarize(text, max_length=int(max_len))
        if method in ("T5", "Compare all"):
            outs["T5"] = get_model("t5").summarize(text, max_length=int(max_len))
        lines = []
        for name, s in outs.items():
            lines.append(f"=== {name} ({len(text.split())} -> {len(s.split())} words) ===\n{s}")
            if reference.strip():
                r = rouge_single(reference, s)
                lines.append(f"ROUGE-1/2/L: {r['rouge1']} / {r['rouge2']} / {r['rougeL']}")
            lines.append("")
        return "\n".join(lines)

    demo = gr.Interface(
        fn=summarize_ui,
        inputs=[
            gr.Textbox(lines=12, label="Paste a news article or paper text"),
            gr.Radio(["TextRank", "BART", "T5", "Compare all"], value="Compare all", label="Method"),
            gr.Slider(1, 10, value=3, step=1, label="TextRank: number of sentences"),
            gr.Slider(40, 250, value=128, step=1, label="Abstractive: max summary tokens"),
            gr.Textbox(lines=3, label="Optional reference summary (for ROUGE)"),
        ],
        outputs=gr.Textbox(lines=14, label="Summary"),
        title="News & Research Paper Summarizer",
        description="Extractive (TextRank) vs Abstractive (BART / T5)",
    )
    demo.launch(share=True)

Device: cuda


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

... (14 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]